In [ ]:
# ==========================================
# IMPROVED PRESIDENTIAL QA CLASSIFIER WITH AWP
# ==========================================
import zipfile
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk
from collections import Counter
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EvalPrediction
)
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
# ==========================================
# 0. UNZIP DATASET
# ==========================================
zip_file_path = "processed_dataset.zip"
extract_to = "./processed_dataset"
if not os.path.exists(extract_to):
    print(f"--- Unzipping {zip_file_path} ---")
    if os.path.exists(zip_file_path):
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"Extracted to {extract_to}")
    else:
        raise FileNotFoundError(f"Please upload '{zip_file_path}' to the current directory before running.")
else:
    print(f"Folder {extract_to} already exists. Skipping unzip.")
# ==========================================
# 1. CONFIGURATION & DEVICE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
MODEL_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
DATASET_PATH = "./processed_dataset"
# Hyperparameters
MAX_LENGTH = 2048
BATCH_SIZE = 16
LEARNING_RATE = 2e-5  # Slightly higher
EPOCHS = 5  # More epochs with better schedule
# AWP Hyperparameters
AWP_EPSILON = 1e-3  # Perturbation magnitude
AWP_START_EPOCH = 1  # Start AWP after first epoch
# Threshold Search Range
THRESH_RANGE = np.arange(-5.0, 5.1, 0.1)
# Labels
LABEL_MAP = {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}
ID2LABEL = {0: 'Clear Reply', 1: 'Ambivalent', 2: 'Clear Non-Reply'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
# ==========================================
# 2. DATA PREPARATION WITH BETTER HYPOTHESIS
# ==========================================
print("--- Loading Processed Data ---")
try:
    dataset = load_from_disk(DATASET_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Could not load data from {DATASET_PATH}. Check unzip step.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.model_max_length = MAX_LENGTH
def preprocess_function(examples):
    """
    IMPROVED HYPOTHESIS: More explicit about what constitutes a "clear" response
    One hypothesis per example that emphasizes:
    - Directness
    - Completeness
    - Specificity
    - Lack of ambiguity
    """
    hypotheses = [
        f"Within this response, the speaker directly answers the specific question '{q}' with clear and complete information, without evasion or ambiguity regarding this particular question."
        for q in examples['formatted_question']
    ]
    premises = examples['interview_answer']
    model_inputs = tokenizer(
        premises,
        hypotheses,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )
    if 'clarity_label' in examples:
        model_inputs["labels"] = [LABEL_MAP[label] for label in examples['clarity_label']]
    return model_inputs
print("--- Tokenizing & Formatting ---")
encoded_dataset = dataset.map(preprocess_function, batched=True)
# ==========================================
# 3. CLASS WEIGHTS WITH FOCAL LOSS
# ==========================================
print("--- Calculating Class Weights ---")
train_labels = encoded_dataset["train"]["labels"]
class_counts = Counter(train_labels)
num_samples = len(train_labels)
num_classes = len(LABEL_MAP)
class_weights = []
for i in range(num_classes):
    weight = num_samples / (num_classes * class_counts[i])
    class_weights.append(weight)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights: {class_weights_tensor}")
# ==========================================
# 4. FOCAL LOSS IMPLEMENTATION
# ==========================================
class FocalLoss(nn.Module):
    """
    Focal Loss focuses training on hard examples
    Helps with class imbalance better than standard weighted CE
    """
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(
            inputs, targets,
            reduction='none',
            weight=self.alpha
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss
# ==========================================
# 4.5 ADVERSARIAL WEIGHT PERTURBATION (AWP)
# ==========================================
class AWP:
    """
    Adversarial Weight Perturbation
    Adds adversarial perturbations to model weights during training
    to improve generalization and robustness
    """
    def __init__(self, model, epsilon=1e-3):
        self.model = model
        self.epsilon = epsilon
        self.backup = {}
        self.backup_eps = {}

    def attack(self):
        """Compute and apply adversarial perturbation"""
        e = 1e-6
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                # Backup original weights
                self.backup[name] = param.data.clone()
                # Compute norm
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    # Compute perturbation
                    perturbation = self.epsilon * param.grad / (norm + e)
                    # Apply perturbation
                    param.data.add_(perturbation)

    def restore(self):
        """Restore original weights"""
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}
# ==========================================
# 5. IMPROVED THRESHOLD LOGIC
# ==========================================
def apply_hierarchical_threshold(logits, thresh_reply, thresh_nonreply):
    """
    REVERSED & IMPROVED LOGIC:
    Default to Ambivalent (most common class)
    Only classify as Reply/NonReply if they BEAT Ambivalent by threshold
    This is more intuitive:
    - Positive threshold = Reply/NonReply needs to be HIGHER than Ambivalent
    - thresh_reply: how much Reply must beat Ambivalent to win
    - thresh_nonreply: how much NonReply must beat Ambivalent to win
    Args:
        logits: (N, 3) array of logits
        thresh_reply: Reply must beat Ambivalent by this much
        thresh_nonreply: NonReply must beat Ambivalent by this much
    Returns:
        predictions: (N,) array of class predictions
    """
    reply_logits = logits[:, 0]
    ambivalent_logits = logits[:, 1]
    nonreply_logits = logits[:, 2]
    # Calculate margins: how much does Reply/NonReply beat Ambivalent?
    reply_margin = reply_logits - ambivalent_logits
    nonreply_margin = nonreply_logits - ambivalent_logits
    # Start with all predictions as Ambivalent (class 1)
    preds = np.ones(len(logits), dtype=int)
    # Reply wins if its margin exceeds threshold
    is_reply = reply_margin > thresh_reply
    preds[is_reply] = 0
    # NonReply wins if its margin exceeds threshold
    is_nonreply = nonreply_margin > thresh_nonreply
    preds[is_nonreply] = 2
    # If both exceed their thresholds, choose the one with higher logit
    both_exceed = is_reply & is_nonreply
    if np.any(both_exceed):
        preds[both_exceed] = np.where(
            reply_logits[both_exceed] > nonreply_logits[both_exceed],
            0, 2
        )
    return preds
# ==========================================
# 6. CUSTOM TRAINER WITH FOCAL LOSS AND AWP (FIXED)
# ==========================================
class FocalLossTrainer(Trainer):
    def __init__(self, *args, awp_epsilon=1e-3, awp_start_epoch=1, **kwargs):
        super().__init__(*args, **kwargs)
        # Initialize Focal Loss with class weights
        self.focal_loss = FocalLoss(
            alpha=class_weights_tensor,
            gamma=2.0  # Focus on hard examples
        )
        # Initialize AWP
        self.awp = AWP(self.model, epsilon=awp_epsilon)
        self.awp_start_epoch = awp_start_epoch
        self.current_epoch = 0

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Move loss function to correct device
        if self.focal_loss.alpha is not None:
            self.focal_loss.alpha = self.focal_loss.alpha.to(logits.device)
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        """
        Fixed training_step to use Accelerator.
        Removes manual scaler manipulation to prevent 'unscale_' errors.
        """
        model.train()
        inputs = self._prepare_inputs(inputs)

        # 1. Standard Forward Pass
        # autocast_smart_context_manager handles fp16/bf16 context automatically
        with self.autocast_smart_context_manager():
            loss = self.compute_loss(model, inputs)

        if self.args.gradient_accumulation_steps > 1:
            loss = loss / self.args.gradient_accumulation_steps

        # 2. Standard Backward Pass
        # Use accelerator.backward() instead of manual scaler.scale().backward()
        # This ensures internal states (like _scale) are correctly set for the optimizer step.
        self.accelerator.backward(loss)

        # 3. AWP Attack & Backward
        if self.current_epoch >= self.awp_start_epoch:
            # Save weights and apply perturbation
            self.awp.attack()

            with self.autocast_smart_context_manager():
                adv_loss = self.compute_loss(model, inputs)

            if self.args.gradient_accumulation_steps > 1:
                adv_loss = adv_loss / self.args.gradient_accumulation_steps

            # Backward pass on adversarial loss
            self.accelerator.backward(adv_loss)

            # Restore original weights
            self.awp.restore()

        return loss.detach()

    def _maybe_log_save_evaluate(self, tr_loss, grad_norm, model, trial, epoch, ignore_keys_for_eval, *args, **kwargs):
            """Track current epoch for AWP"""
            self.current_epoch = epoch
            return super()._maybe_log_save_evaluate(
                tr_loss, grad_norm, model, trial, epoch, ignore_keys_for_eval, *args, **kwargs
            )
# ==========================================
# 7. COMPUTE METRICS WITH GRID SEARCH
# ==========================================
def compute_metrics(p: EvalPrediction):
    """
    Performs grid search to find optimal thresholds during evaluation
    """
    logits = p.predictions
    labels = p.label_ids
    best_f1 = -1.0
    best_tr = 0.0
    best_tnr = 0.0
    # Grid Search
    for tr in THRESH_RANGE:
        for tnr in THRESH_RANGE:
            preds = apply_hierarchical_threshold(logits, tr, tnr)
            f1 = f1_score(labels, preds, average='macro')
            if f1 > best_f1:
                best_f1 = f1
                best_tr = tr
                best_tnr = tnr
    # Final metrics with best thresholds
    final_preds = apply_hierarchical_threshold(logits, best_tr, best_tnr)
    acc = accuracy_score(labels, final_preds)
    print(f"\n[Eval] Best Thresh Reply: {best_tr:.2f} | Best Thresh NonReply: {best_tnr:.2f} | F1: {best_f1:.4f}")
    return {
        "f1_macro_best": best_f1,
        "accuracy": acc,
        "best_thresh_reply": best_tr,
        "best_thresh_nonreply": best_tnr
    }
# ==========================================
# 8. MODEL INITIALIZATION
# ==========================================
print("--- Initializing Model ---")
config = AutoConfig.from_pretrained(MODEL_ID)
config.max_position_embeddings = MAX_LENGTH
config.num_labels = 3
config.id2label = ID2LABEL
config.label2id = LABEL2ID
config.use_cache = False
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    config=config,
    ignore_mismatched_sizes=True
)
# ==========================================
# 9. TRAINING ARGUMENTS WITH IMPROVEMENTS
# ==========================================
training_args = TrainingArguments(
    output_dir="./deberta_improved_v2",
    # Learning rate & schedule
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",  # Cosine annealing
    warmup_ratio=0.1,  # 10% warmup
    # Batch settings
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    # Training duration
    num_train_epochs=EPOCHS,
    # Regularization
    weight_decay=0.01,
    label_smoothing_factor=0.1,  # Prevent overconfidence
    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro_best",
    greater_is_better=True,
    save_total_limit=2,
    # Performance optimizations
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Logging
    logging_steps=25,
    report_to="none"
)
# ==========================================
# 10. INITIALIZE TRAINER WITH AWP
# ==========================================
trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    awp_epsilon=AWP_EPSILON,
    awp_start_epoch=AWP_START_EPOCH,
)
# ==========================================
# 11. TRAIN MODEL
# ==========================================
print(f"\n{'='*60}")
print(f"STARTING TRAINING - {EPOCHS} EPOCHS")
print(f"AWP enabled from epoch {AWP_START_EPOCH} with epsilon={AWP_EPSILON}")
print(f"{'='*60}\n")
trainer.train()
# ==========================================
# 12. FINAL EVALUATION WITH DETAILED ANALYSIS
# ==========================================
print("\n" + "="*60)
print("FINAL EVALUATION ON TEST SET")
print("="*60)
predictions_output = trainer.predict(encoded_dataset["test"])
raw_logits = predictions_output.predictions
y_true = predictions_output.label_ids
# Final grid search on test set
final_best_f1 = -1
final_best_tr = 0.0
final_best_tnr = 0.0
f1_matrix = np.zeros((len(THRESH_RANGE), len(THRESH_RANGE)))
print("\n--- Running Final Grid Search ---")
for i, tr in enumerate(THRESH_RANGE):
    for j, tnr in enumerate(THRESH_RANGE):
        temp_preds = apply_hierarchical_threshold(raw_logits, tr, tnr)
        temp_f1 = f1_score(y_true, temp_preds, average='macro')
        f1_matrix[i, j] = temp_f1
        if temp_f1 > final_best_f1:
            final_best_f1 = temp_f1
            final_best_tr = tr
            final_best_tnr = tnr
# Generate final predictions
final_preds = apply_hierarchical_threshold(raw_logits, final_best_tr, final_best_tnr)
final_acc = accuracy_score(y_true, final_preds)
# ==========================================
# 13. RESULTS & INTERPRETATION
# ==========================================
print("\n" + "="*60)
print("OPTIMAL THRESHOLD CONFIGURATION")
print("="*60)
print(f"Reply needs to beat Ambivalent by:     {final_best_tr:.2f}")
print(f"NonReply needs to beat Ambivalent by:  {final_best_tnr:.2f}")
print(f"\nMAX F1-MACRO: {final_best_f1:.4f}")
print(f"ACCURACY:     {final_acc:.4f}")
# Detailed classification report
target_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]
report = classification_report(y_true, final_preds, target_names=target_names, digits=4)
print(f"\n{'='*60}")
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
print(report)
# ==========================================
# 14. LOGIT ANALYSIS
# ==========================================
print("\n" + "="*60)
print("LOGIT STATISTICS BY TRUE CLASS")
print("="*60)
for class_id, class_name in ID2LABEL.items():
    mask = y_true == class_id
    n = mask.sum()
    if n > 0:
        class_logits = raw_logits[mask]
        mean_reply = class_logits[:, 0].mean()
        mean_amb = class_logits[:, 1].mean()
        mean_nonreply = class_logits[:, 2].mean()
        mean_margin_reply = (class_logits[:, 0] - class_logits[:, 1]).mean()
        mean_margin_nonreply = (class_logits[:, 2] - class_logits[:, 1]).mean()
        std_margin_reply = (class_logits[:, 0] - class_logits[:, 1]).std()
        std_margin_nonreply = (class_logits[:, 2] - class_logits[:, 1]).std()
        print(f"\n[{class_name}] (n={n})")
        print(f"  Mean Logits:   Reply={mean_reply:.2f}, Amb={mean_amb:.2f}, NonReply={mean_nonreply:.2f}")
        print(f"  Mean Margins:  Reply-Amb={mean_margin_reply:.2f}, NonReply-Amb={mean_margin_nonreply:.2f}")
        print(f"  Std Margins:   Reply-Amb={std_margin_reply:.2f}, NonReply-Amb={std_margin_nonreply:.2f}")
# ==========================================
# 15. VISUALIZATIONS
# ==========================================
print("\n--- Generating Visualizations ---")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Plot 1: Confusion Matrix
cm = confusion_matrix(y_true, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names,
            yticklabels=target_names,
            ax=axes[0])
axes[0].set_title(f'Confusion Matrix\n(Tr={final_best_tr:.2f}, Tnr={final_best_tnr:.2f})', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('True', fontsize=11)
# Plot 2: Grid Search Heatmap
im = axes[1].imshow(f1_matrix, cmap='viridis', aspect='auto', origin='lower')
axes[1].set_title('Grid Search: F1 Score Landscape', fontsize=12)
axes[1].set_xlabel('Threshold NonReply (Low → High)', fontsize=11)
axes[1].set_ylabel('Threshold Reply (Low → High)', fontsize=11)
# Add colorbar
cbar = plt.colorbar(im, ax=axes[1])
cbar.set_label('F1 Macro Score', fontsize=10)
# Mark optimal point
opt_i = np.argmin(np.abs(THRESH_RANGE - final_best_tr))
opt_j = np.argmin(np.abs(THRESH_RANGE - final_best_tnr))
axes[1].plot(opt_j, opt_i, 'r*', markersize=20, label='Optimal')
axes[1].legend()
plt.tight_layout()
plt.savefig('classification_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"Results saved to: classification_results.png")
print(f"Model saved to: ./deberta_improved_v2")

In [ ]:
import os
import shutil
import json
from google.colab import drive

# ==========================================
# 1. MOUNT GOOGLE DRIVE
# ==========================================
drive.mount('/content/drive')

# Define where you want to save it in Drive
# Change "Presidential_QA_Best_Model" to whatever folder name you prefer
DRIVE_DEST_PATH = "/content/drive/My Drive/Presidential_QA_Best_Model_76"

# Create the directory in Drive if it doesn't exist
if not os.path.exists(DRIVE_DEST_PATH):
    os.makedirs(DRIVE_DEST_PATH)
    print(f"Created Drive folder: {DRIVE_DEST_PATH}")

# ==========================================
# 2. SAVE MODEL & TOKENIZER
# ==========================================
print("--- Saving Best Model & Tokenizer ---")
# We save to a temporary local folder first to ensure we package the tokenizer and model together cleanly
local_save_path = "./temp_best_model"

# trainer.save_model() saves the model currently active in the trainer
# Since load_best_model_at_end=True, this is your BEST model
trainer.save_model(local_save_path)
tokenizer.save_pretrained(local_save_path)

# Copy the model folder to Drive
drive_model_path = os.path.join(DRIVE_DEST_PATH, "model")
if os.path.exists(drive_model_path):
    shutil.rmtree(drive_model_path) # Clean overwrite
shutil.copytree(local_save_path, drive_model_path)
print(f"Model saved to: {drive_model_path}")

# ==========================================
# 3. SAVE THRESHOLD STATS
# ==========================================
print("--- Saving Threshold Statistics ---")
threshold_data = {
    "best_thresh_reply": final_best_tr,
    "best_thresh_nonreply": final_best_tnr,
    "best_f1_score": final_best_f1,
    "accuracy": final_acc,
    "model_name": MODEL_ID
}

# Save as JSON locally
json_path = "threshold_config.json"
with open(json_path, 'w') as f:
    json.dump(threshold_data, f, indent=4)

# Copy JSON to Drive
shutil.copy(json_path, os.path.join(DRIVE_DEST_PATH, json_path))
print(f"Threshold config saved to: {os.path.join(DRIVE_DEST_PATH, json_path)}")

# ==========================================
# 4. SAVE PLOTS
# ==========================================
# Copy the results image if it exists
img_name = "classification_results.png"
if os.path.exists(img_name):
    shutil.copy(img_name, os.path.join(DRIVE_DEST_PATH, img_name))
    print(f"Plot saved to: {os.path.join(DRIVE_DEST_PATH, img_name)}")

print(f"\nSUCCESS: All files saved to {DRIVE_DEST_PATH}")